# Molecular Graph Learning Curve

This notebook measures how predictive performance on the MoleculeNet HIV dataset changes as the number of streamed training graphs increases, using `NSPPK` features with `radius=1`, `distance=4`, `connector=1` and a selectable incremental classifier (`SGDClassifier`, `MLPClassifier`, or any custom estimator that supports `partial_fit(...)`).

The first `TEST_PREFIX_SIZE` molecules are materialized, balanced, and used as a fixed test set. Training is then repeated on random Bernoulli samples from the remaining molecules with `LIMIT = 0.9`, so each repeat sees a different large training stream while the same held-out test set is scored at exponentially growing train sizes. Because the default HIV setup does not require learned attribute preprocessing, the benchmark can stream labeled batches directly without a separate warmup fit phase.


In [ ]:
from pathlib import Path
import csv
import gc
import os
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rdkit import RDLogger
from sklearn.base import clone
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.neural_network import MLPClassifier

try:
    import psutil
except ImportError:
    psutil = None

REPO_CANDIDATES = [Path.cwd().resolve(), Path.cwd().resolve().parent]
REPO_ROOT = next(candidate for candidate in REPO_CANDIDATES if (candidate / 'src').exists())
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from nsppk import NSPPK
from utils import plot_series_with_band_loess

DATASET_FILE = REPO_ROOT / 'data' / 'HIV.csv'
NBIT = 12
RADIUS = 1
DISTANCE = 4
CONNECTOR = 1
LIMIT = 0.99
TEST_PREFIX_SIZE = 2000
BALANCE_TEST_SET = True
TRAIN_SIZE_VALUES = None
N_REPEATS = 3
BATCH_SIZE = 256
PARALLEL = True
RANDOM_STATE = 42
CLASSIFIER_CHOICE = 'sgd'  # 'sgd', 'mlp', or 'generic'
CLASSIFIER_PARAMS = {} #{'hidden_layer_sizes': (256,64,)}
GENERIC_PARTIAL_FIT_CLASSIFIER = None
CLASSIFIER_REQUIRES_DENSE = CLASSIFIER_CHOICE == 'mlp'

RDLogger.DisableLog('rdApp.*')

def current_rss_mb():
    if psutil is not None:
        return psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2)
    import resource
    rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    if sys.platform == 'darwin':
        return rss / (1024 ** 2)
    return rss / 1024

MEMORY_LOG = []
MEMORY_PEAK_MB = 0.0

def record_memory(stage, emit=True):
    global MEMORY_PEAK_MB
    rss_mb = float(current_rss_mb())
    MEMORY_PEAK_MB = max(MEMORY_PEAK_MB, rss_mb)
    event = {'stage': stage, 'rss_mb': rss_mb, 'peak_rss_mb': MEMORY_PEAK_MB}
    MEMORY_LOG.append(event)
    if emit:
        print(f"[memory] {stage:<28} rss={rss_mb:8.2f} MB  peak={MEMORY_PEAK_MB:8.2f} MB")
    return event

def count_dataset_rows(dataset_file):
    with Path(dataset_file).open() as stream:
        header_seen = False
        count = 0
        for line in stream:
            if not line.strip():
                continue
            if not header_seen:
                header_seen = True
                continue
            count += 1
        return count

def resolve_limit_count(limit, available_count):
    if limit is None:
        return int(available_count)
    if isinstance(limit, int):
        return min(int(limit), int(available_count))
    return min(int(np.floor(float(limit) * available_count)), int(available_count))

def make_halving_train_sizes(dataset_size, train_limit, min_size=BATCH_SIZE):
    if train_limit <= 0:
        return []
    if TRAIN_SIZE_VALUES is not None:
        train_sizes = [n for n in TRAIN_SIZE_VALUES if n <= train_limit]
        if train_limit not in train_sizes:
            train_sizes.append(train_limit)
        return sorted(set(train_sizes))
    train_sizes = []
    current = dataset_size // 2
    while current >= min_size:
        if current <= train_limit:
            train_sizes.append(current)
        current //= 2
    train_sizes.append(train_limit)
    return sorted(set(train_sizes))

def compute_balanced_class_weight(labels, classes):
    labels = np.asarray(labels)
    classes = np.asarray(classes)
    n_samples = labels.shape[0]
    n_classes = classes.shape[0]
    weights = {}
    for cls in classes:
        class_count = int(np.sum(labels == cls))
        if class_count == 0:
            raise ValueError(f'Cannot compute class weight: class {cls} is absent from the reference labels.')
        weights[int(cls)] = n_samples / (n_classes * class_count)
    return weights

def load_labels_from_csv(dataset_file, label_column, start_after_instance=0):
    labels = []
    with Path(dataset_file).open(newline='') as stream:
        reader = csv.DictReader(line for line in stream if line.strip())
        for row_idx, row in enumerate(reader):
            if row_idx < start_after_instance:
                continue
            labels.append(int(row[label_column]))
    return np.asarray(labels)

def build_partial_fit_classifier(choice, random_state, class_weight=None, classifier_params=None, generic_classifier=None):
    classifier_params = dict(classifier_params or {})
    if choice == 'sgd':
        default_params = {
            'loss': 'log_loss',
            'alpha': 1e-5,
            'penalty': 'l2',
            'random_state': random_state,
            'class_weight': class_weight,
        }
        default_params.update(classifier_params)
        classifier = SGDClassifier(**default_params)
    elif choice == 'mlp':
        if class_weight is not None:
            raise ValueError('MLPClassifier does not accept class_weight; set CLASSIFIER_CHOICE to \'sgd\' or use a generic classifier instead.')
        default_params = {
            'hidden_layer_sizes': (256,),
            'alpha': 1e-4,
            'learning_rate_init': 1e-3,
            'max_iter': 1,
            'random_state': random_state,
        }
        default_params.update(classifier_params)
        classifier = MLPClassifier(**default_params)
    elif choice == 'generic':
        if generic_classifier is None:
            raise ValueError('Set GENERIC_PARTIAL_FIT_CLASSIFIER to an estimator instance or zero-argument factory when CLASSIFIER_CHOICE="generic".')
        classifier = generic_classifier() if callable(generic_classifier) else clone(generic_classifier)
    else:
        raise ValueError(f'Unknown CLASSIFIER_CHOICE: {choice!r}')

    if not hasattr(classifier, 'partial_fit'):
        raise TypeError(f'{type(classifier).__name__} does not implement partial_fit(...)')
    return classifier

def get_positive_scores(classifier, X):
    if hasattr(classifier, 'decision_function'):
        scores = classifier.decision_function(X)
    elif hasattr(classifier, 'predict_proba'):
        scores = classifier.predict_proba(X)
    else:
        raise TypeError(f'{type(classifier).__name__} does not expose decision_function(...) or predict_proba(...) for scoring')

    scores = np.asarray(scores)
    if scores.ndim == 2:
        if scores.shape[1] < 2:
            raise ValueError(f'Expected binary-class scores with at least 2 columns, got shape {scores.shape}')
        scores = scores[:, 1]
    return scores.ravel()


## Setup

Build the fixed balanced test set from the first `TEST_PREFIX_SIZE` molecules, instantiate the vectorizer directly, and convert the fractional `LIMIT` into an approximate sampled training-pool size. The checkpoint schedule is then derived by repeatedly halving the total dataset size down to a minimum of one `BATCH_SIZE`, keeping only values that fit within the sampled training cap.


In [ ]:
loader = NSPPK(radius=RADIUS, distance=DISTANCE, connector=CONNECTOR, nbits=NBIT, parallel=PARALLEL)
record_memory('start')

test_graphs = loader.load_from(
    DATASET_FILE,
    'smiles',
    limit=TEST_PREFIX_SIZE,
    balance=BALANCE_TEST_SET,
    random_state=RANDOM_STATE,
    label_extractor=lambda graph: int(graph.graph['HIV_active']),
)
y_test = np.asarray([int(graph.graph['HIV_active']) for graph in test_graphs])
record_memory('test set loaded')

dataset_size = count_dataset_rows(DATASET_FILE)
available_train_pool = max(dataset_size - TEST_PREFIX_SIZE, 0)
train_limit = resolve_limit_count(LIMIT, available_train_pool)

vectorizer = NSPPK(
    radius=RADIUS,
    distance=DISTANCE,
    connector=CONNECTOR,
    nbits=NBIT,
    dense=CLASSIFIER_REQUIRES_DENSE,
    parallel=PARALLEL,
)
record_memory('vectorizer ready')
X_test = vectorizer.transform(test_graphs)
record_memory('test transformed')

classes = np.array([0, 1])
train_pool_labels = load_labels_from_csv(DATASET_FILE, 'HIV_active', start_after_instance=TEST_PREFIX_SIZE)
class_weight = compute_balanced_class_weight(train_pool_labels, classes)
classifier_label = CLASSIFIER_CHOICE if CLASSIFIER_CHOICE != 'generic' else type(GENERIC_PARTIAL_FIT_CLASSIFIER).__name__ if GENERIC_PARTIAL_FIT_CLASSIFIER is not None and not callable(GENERIC_PARTIAL_FIT_CLASSIFIER) else 'generic'

train_size_values = make_halving_train_sizes(dataset_size, train_limit)

print('test graphs:', len(test_graphs))
print(f'test positive rate: {y_test.mean():.3f}')
print('dataset size:', dataset_size)
print('available train pool:', available_train_pool)
print('train pool limit:', train_limit)
print('classifier choice:', classifier_label)
print('class weight for partial_fit:', class_weight)
print('classifier requires dense input:', CLASSIFIER_REQUIRES_DENSE)
print('stream batch size:', BATCH_SIZE)
print('train sizes:', train_size_values)
print(f'initial peak RSS: {MEMORY_PEAK_MB:.2f} MB')


## Incremental Learning Curve

Train the selected incremental classifier with `partial_fit(...)` on successive labeled batches drawn from a repeat-specific Bernoulli sample of the post-test dataset. The first streamed batch initializes `partial_fit(...)` with the full class list, and later batches continue incrementally. Metrics and memory are recorded whenever the cumulative trained count crosses one of the power-of-two checkpoints generated from the sampled training limit.


In [ ]:
results = []

print(f"{'train':>8} | {'repeat':>6} | {'roc_auc':>8} | {'avg_prec':>8} | {'seconds':>8} | {'rss_mb':>8} | {'peak_mb':>8}")
print('-' * 79)

repeat_histories = {
    train_size: {'roc_auc': [], 'avg_precision': [], 'runtime_sec': [], 'rss_mb': [], 'peak_rss_mb': []}
    for train_size in train_size_values
}
final_repeat_history = {
    'train_size': [],
    'roc_auc': [],
    'avg_precision': [],
    'runtime_sec': [],
    'rss_mb': [],
    'peak_rss_mb': [],
}

def ensure_history(train_size):
    if train_size not in repeat_histories:
        repeat_histories[train_size] = {
            'roc_auc': [],
            'avg_precision': [],
            'runtime_sec': [],
            'rss_mb': [],
            'peak_rss_mb': [],
        }

def record_checkpoint(train_size, classifier, repeat_idx, t0, history=None):
    if history is None:
        ensure_history(train_size)
        history = repeat_histories[train_size]
    memory_event = record_memory(f'repeat {repeat_idx} train {train_size}', emit=False)
    y_score = get_positive_scores(classifier, X_test)
    runtime_sec = time.perf_counter() - t0
    roc_auc = roc_auc_score(y_test, y_score)
    avg_precision = average_precision_score(y_test, y_score)

    if 'train_size' in history:
        history['train_size'].append(int(train_size))
    history['roc_auc'].append(float(roc_auc))
    history['avg_precision'].append(float(avg_precision))
    history['runtime_sec'].append(float(runtime_sec))
    history['rss_mb'].append(float(memory_event['rss_mb']))
    history['peak_rss_mb'].append(float(memory_event['peak_rss_mb']))

    print(
        f"{train_size:>8d} | {repeat_idx:>6d} | {roc_auc:>8.4f} | {avg_precision:>8.4f} | "
        f"{runtime_sec:>8.2f} | {memory_event['rss_mb']:>8.2f} | {memory_event['peak_rss_mb']:>8.2f}"
    )

for repeat_idx in range(1, N_REPEATS + 1):
    classifier = build_partial_fit_classifier(
        CLASSIFIER_CHOICE,
        random_state=RANDOM_STATE + repeat_idx,
        class_weight=class_weight,
        classifier_params=CLASSIFIER_PARAMS,
        generic_classifier=GENERIC_PARTIAL_FIT_CLASSIFIER,
    )

    t0 = time.perf_counter()
    trained_count = 0
    checkpoint_idx = 0
    last_recorded_train_size = [None]
    is_first_batch = True

    def score_reached_checkpoints(trained_count, checkpoint_idx):
        while checkpoint_idx < len(train_size_values) and trained_count >= train_size_values[checkpoint_idx]:
            stop = train_size_values[checkpoint_idx]
            record_checkpoint(stop, classifier, repeat_idx, t0)
            last_recorded_train_size[0] = stop
            checkpoint_idx += 1

        return checkpoint_idx

    for X_batch, y_batch in vectorizer.stream_from(
        DATASET_FILE,
        'smiles',
        limit=LIMIT,
        random_state=RANDOM_STATE + repeat_idx,
        batch_size=BATCH_SIZE,
        label_extractor=lambda graph: int(graph.graph['HIV_active']),
        start_after_instance=TEST_PREFIX_SIZE,
    ):
        if is_first_batch:
            classifier.partial_fit(X_batch, y_batch, classes=classes)
            is_first_batch = False
        else:
            classifier.partial_fit(X_batch, y_batch)
        trained_count += len(y_batch)
        checkpoint_idx = score_reached_checkpoints(trained_count, checkpoint_idx)
        del X_batch, y_batch
        gc.collect()

    if trained_count > 0 and trained_count != last_recorded_train_size[0]:
        record_checkpoint(trained_count, classifier, repeat_idx, t0, history=final_repeat_history)

    print(f'repeat {repeat_idx} sampled train graphs: {trained_count}')

reached_train_sizes = []
skipped_train_sizes = []
for train_size in sorted(repeat_histories):
    history = repeat_histories[train_size]
    if not history['roc_auc']:
        if train_size == train_limit and final_repeat_history['roc_auc']:
            continue
        skipped_train_sizes.append(train_size)
        continue
    reached_train_sizes.append(train_size)
    results.append(
        {
            'train_size': train_size,
            'n_repeats_reached': len(history['roc_auc']),
            'mean_roc_auc': float(np.mean(history['roc_auc'])),
            'std_roc_auc': float(np.std(history['roc_auc'], ddof=0)),
            'mean_avg_precision': float(np.mean(history['avg_precision'])),
            'std_avg_precision': float(np.std(history['avg_precision'], ddof=0)),
            'mean_runtime_sec': float(np.mean(history['runtime_sec'])),
            'std_runtime_sec': float(np.std(history['runtime_sec'], ddof=0)),
            'mean_rss_mb': float(np.mean(history['rss_mb'])),
            'mean_peak_rss_mb': float(np.mean(history['peak_rss_mb'])),
            'classifier': classifier_label,
            'nbits': NBIT,
            'radius': RADIUS,
            'distance': DISTANCE,
            'connector': CONNECTOR,
        }
    )

if final_repeat_history['roc_auc']:
    final_train_size = int(round(np.mean(final_repeat_history['train_size'])))
    reached_train_sizes.append(final_train_size)
    results.append(
        {
            'train_size': final_train_size,
            'n_repeats_reached': len(final_repeat_history['roc_auc']),
            'mean_roc_auc': float(np.mean(final_repeat_history['roc_auc'])),
            'std_roc_auc': float(np.std(final_repeat_history['roc_auc'], ddof=0)),
            'mean_avg_precision': float(np.mean(final_repeat_history['avg_precision'])),
            'std_avg_precision': float(np.std(final_repeat_history['avg_precision'], ddof=0)),
            'mean_runtime_sec': float(np.mean(final_repeat_history['runtime_sec'])),
            'std_runtime_sec': float(np.std(final_repeat_history['runtime_sec'], ddof=0)),
            'mean_rss_mb': float(np.mean(final_repeat_history['rss_mb'])),
            'mean_peak_rss_mb': float(np.mean(final_repeat_history['peak_rss_mb'])),
            'classifier': classifier_label,
            'nbits': NBIT,
            'radius': RADIUS,
            'distance': DISTANCE,
            'connector': CONNECTOR,
        }
    )

results_df = pd.DataFrame(results)
if skipped_train_sizes:
    print('skipped train sizes with no reached repeats:', skipped_train_sizes)
print('summarized train sizes:', reached_train_sizes)
print(f'final peak RSS: {MEMORY_PEAK_MB:.2f} MB')
results_df


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
frac = 0.8  # Local quadratic LOESS span; adjust between 0 and 1 for more or less smoothing.
x = results_df['train_size'].to_numpy()

plot_series_with_band_loess(
    axes[0],
    x,
    results_df['mean_roc_auc'],
    y_std=results_df['std_roc_auc'],
    frac=frac,
    label='Mean ROC-AUC',
)
axes[0].set_xlabel('Training graphs')
axes[0].set_ylabel('ROC-AUC')
axes[0].set_title('Learning curve: ROC-AUC')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

plot_series_with_band_loess(
    axes[1],
    x,
    results_df['mean_avg_precision'],
    y_std=results_df['std_avg_precision'],
    frac=frac,
    color='tab:green',
    label='Mean average precision',
)
axes[1].set_xlabel('Training graphs')
axes[1].set_ylabel('Average precision')
axes[1].set_title('Learning curve: average precision')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plot_series_with_band_loess(
    axes[2],
    x,
    results_df['mean_runtime_sec'],
    y_std=results_df['std_runtime_sec'],
    frac=frac,
    color='tab:red',
    label='Mean runtime',
)
axes[2].set_xlabel('Training graphs')
axes[2].set_ylabel('Seconds')
axes[2].set_title('End-to-end runtime')
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()


## Minimal Full-Stream `partial_fit` Example

For the default HIV setup, `NSPPK.fit(...)` is a no-op because no attribute embedding or clustering must be learned. That means `stream_from(...)` can directly yield supervised `(X_batch, y_batch)` pairs when a `label_extractor` is provided.


In [ ]:
%%time

stream_vectorizer = NSPPK(
    radius=RADIUS,
    distance=DISTANCE,
    connector=CONNECTOR,
    nbits=NBIT,
    dense=CLASSIFIER_REQUIRES_DENSE,
    parallel=PARALLEL,
)

stream_model = build_partial_fit_classifier(
    CLASSIFIER_CHOICE,
    random_state=RANDOM_STATE,
    class_weight=class_weight,
    classifier_params=CLASSIFIER_PARAMS,
    generic_classifier=GENERIC_PARTIAL_FIT_CLASSIFIER,
)
classes = np.array([0, 1])

for batch_idx, (X_batch, y_batch) in enumerate(
    stream_vectorizer.stream_from(
        DATASET_FILE,
        'smiles',
        batch_size=BATCH_SIZE,
        label_extractor=lambda graph: int(graph.graph['HIV_active']),
    )
):
    if batch_idx == 0:
        stream_model.partial_fit(X_batch, y_batch, classes=classes)
    else:
        stream_model.partial_fit(X_batch, y_batch)

stream_model
